# 5. Human + Agent Review of Low-Confidence Predictions

Two parts:
1. **Agent reviewer.** Audits low-confidence rows in `cat_predictions`
   against the active taxonomy. Now uses the **generic**
   `TaxonomyReviewAgent` so it works for any schema (three_level,
   gl_map, unspsc). Writes its own row per audit with
   `source='agent_review'`.
2. **Sync to Lakebase.** Mirrors `cat_predictions`, taxonomies, the
   schema registry, and `invoices` into Lakebase Postgres so the React
   app can serve them with low latency. Creates the writable native
   `cat_reviews` table for human corrections.

Tested on Serverless v4.

In [ ]:
%pip install pyyaml openai openpyxl pydantic 'databricks-sdk>=0.50.0' psycopg2-binary
%restart_python

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))

from src.utils import get_spark
from src.config import load_config
from src.schemas import get_schema
from src.agents.tools import TaxonomyToolset
from src.agents.review_agent import TaxonomyReviewAgent
from src.confidence import normalize_llm_self_report
from databricks.sdk import WorkspaceClient
import pyspark.sql.functions as F
import concurrent.futures
from datetime import datetime

spark = get_spark()
config = load_config()

In [ ]:
dbutils.widgets.removeAll()
dbutils.widgets.text('catalog', config.catalog)
dbutils.widgets.text('schema', config.schema_name)
dbutils.widgets.text('llm_endpoint', config.large_llm_endpoint)
dbutils.widgets.text('review_schema', 'three_level')
dbutils.widgets.text('review_source', 'ai_classify')
dbutils.widgets.text('low_confidence_threshold', '0.6')
dbutils.widgets.text('sample_size', '50')
dbutils.widgets.text('lakebase_instance', config.lakebase_instance)

catalog = dbutils.widgets.get('catalog')
schema = dbutils.widgets.get('schema')
llm_endpoint = dbutils.widgets.get('llm_endpoint')
review_schema = dbutils.widgets.get('review_schema')
review_source = dbutils.widgets.get('review_source')
thresh = float(dbutils.widgets.get('low_confidence_threshold'))
sample_size = int(dbutils.widgets.get('sample_size'))
instance_name = dbutils.widgets.get('lakebase_instance')
spark.sql(f'USE {catalog}.{schema}')

## 1. Agent review of low-confidence predictions

In [ ]:
candidates = spark.sql(f"""
  SELECT p.order_id,
         CONCAT_WS(' | ', i.supplier, i.description, i.cost_centre) AS invoice_info,
         p.code AS bootstrap_code,
         p.label AS bootstrap_label,
         p.level_path AS bootstrap_level_path,
         p.confidence,
         p.rationale
  FROM {catalog}.{schema}.cat_predictions p
  JOIN {catalog}.{schema}.invoices i ON i.order_id = p.order_id
  WHERE p.schema_name = '{review_schema}' AND p.source = '{review_source}'
    AND (p.confidence IS NULL OR p.confidence < {thresh})
  ORDER BY p.confidence ASC NULLS FIRST
  LIMIT {sample_size}
""").toPandas()
print(f'Reviewing {len(candidates)} rows for schema={review_schema} source={review_source}')

# Generic taxonomy: pulled straight from the normalized taxonomy table.
# The TaxonomyToolset only requires {code, label, level_path} columns.
taxonomy_pdf = spark.table(f'{catalog}.{schema}.{review_schema}_taxonomy').toPandas()
display_name = get_schema(review_schema).display_name

In [ ]:
if len(candidates) > 0:
    openai_client = WorkspaceClient().serving_endpoints.get_open_ai_client()
    toolset = TaxonomyToolset(taxonomy=taxonomy_pdf)
    agent = TaxonomyReviewAgent(
        openai_client=openai_client,
        model=llm_endpoint,
        toolset=toolset,
        schema_display_name=display_name,
        max_steps=12,
    )

    def _review_one(row):
        path = list(row.get('bootstrap_level_path') or [])
        return agent.review(
            order_id=str(row['order_id']),
            invoice_text=str(row.get('invoice_info', '')),
            bootstrap_code=str(row.get('bootstrap_code')) if row.get('bootstrap_code') else None,
            bootstrap_label=str(row.get('bootstrap_label')) if row.get('bootstrap_label') else None,
            bootstrap_level_path=path,
            bootstrap_confidence=row.get('confidence'),
            bootstrap_rationale=row.get('rationale'),
        )

    work = candidates.to_dict(orient='records')
    results = []
    with concurrent.futures.ThreadPoolExecutor(max_workers=8) as ex:
        for r in ex.map(_review_one, work):
            results.append(r)
    print(f'Agent reviewed {len(results)} rows')
else:
    results = []
    print('No low-confidence rows to review.')

In [ ]:
if results:
    from pyspark.sql.types import (StructType, StructField, StringType,
                                   ArrayType, DoubleType, TimestampType)
    pred_schema = StructType([
        StructField('order_id', StringType()), StructField('schema_name', StringType()),
        StructField('code', StringType()), StructField('label', StringType()),
        StructField('level_path', ArrayType(StringType())),
        StructField('confidence', DoubleType()), StructField('rationale', StringType()),
        StructField('source', StringType()),
        StructField('candidates', ArrayType(StructType([
            StructField('code', StringType()), StructField('label', StringType()),
            StructField('score', DoubleType()),
        ]))),
        StructField('classified_at', TimestampType()),
    ])
    rows = []
    now = datetime.utcnow()
    for r in results:
        if r.error and r.error not in ('max_steps_exceeded_fallback_to_bootstrap',):
            continue
        path = list(r.suggested_level_path or [])
        code = r.suggested_code or (path[-1] if path else None)
        label = r.suggested_label or (path[-1] if path else None)
        if code is None:
            continue
        # Normalize agent's 1-5 confidence with the unified helper.
        conf = normalize_llm_self_report(r.agent_confidence) if r.agent_confidence else None
        rows.append({
            'order_id': r.order_id,
            'schema_name': review_schema,
            'code': str(code),
            'label': str(label),
            'level_path': [str(p) for p in path],
            'confidence': conf,
            'rationale': r.rationale or '',
            'source': 'agent_review',
            'candidates': [],
            'classified_at': now,
        })
    if rows:
        sdf = spark.createDataFrame(rows, schema=pred_schema)
        sdf.createOrReplaceTempView('_agent_review')
        spark.sql(f"""
        MERGE INTO {catalog}.{schema}.cat_predictions t
        USING _agent_review s
        ON t.order_id = s.order_id AND t.schema_name = s.schema_name AND t.source = s.source
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
        """)
        print(f'Merged {len(rows)} agent_review predictions into cat_predictions')
    else:
        print('No agent results to merge.')

## 2. Sync to Lakebase

Mirrors the unified tables and creates the writable `cat_reviews` table
the app POSTs human corrections into.

In [ ]:
from databricks.sdk.service.database import (
    SyncedDatabaseTable, SyncedTableSpec, NewPipelineSpec,
    SyncedTableSchedulingPolicy,
)
import psycopg2, uuid

w = WorkspaceClient()
dbname = config.lakebase_dbname

# unspsc_taxonomy is large (~150k rows) but worth syncing: lets the app
# read the hierarchy directly from Lakebase (Postgres) instead of falling
# back to parsing the bundled Excel asset on first request.
TABLES_TO_SYNC = [
    {'source': 'invoices', 'pk': ['order_id']},
    {'source': 'cat_predictions', 'pk': ['order_id', 'schema_name', 'source']},
    {'source': 'schema_registry', 'pk': ['name']},
    {'source': 'three_level_taxonomy', 'pk': ['code']},
    {'source': 'gl_map_taxonomy', 'pk': ['code']},
    {'source': 'unspsc_taxonomy', 'pk': ['code']},
]

for t in TABLES_TO_SYNC:
    src_full = f'{catalog}.{schema}.{t["source"]}'
    sync_full = f'{catalog}.{schema}.{t["source"]}_synced'
    print(f'Syncing {src_full} -> {sync_full}')
    try:
        existing = w.database.get_synced_database_table(name=sync_full)
        print(f'  exists ({existing.data_synchronization_status.detailed_state})')
        continue
    except Exception:
        pass
    try:
        w.database.create_synced_database_table(
            SyncedDatabaseTable(
                name=sync_full,
                database_instance_name=instance_name,
                logical_database_name=dbname,
                spec=SyncedTableSpec(
                    source_table_full_name=src_full,
                    primary_key_columns=t['pk'],
                    scheduling_policy=SyncedTableSchedulingPolicy.SNAPSHOT,
                    create_database_objects_if_missing=True,
                    new_pipeline_spec=NewPipelineSpec(storage_catalog=catalog, storage_schema=schema),
                ),
            )
        )
        print(f'  created')
    except Exception as e:
        print(f'  error: {e}')


In [ ]:
# Create writable cat_reviews
db = w.database.get_database_instance(name=instance_name)
cred = w.database.generate_database_credential(
    request_id=str(uuid.uuid4()), instance_names=[instance_name]
)
me = w.current_user.me()
user_email = me.emails[0].value if me.emails else me.user_name
conn = psycopg2.connect(host=db.read_write_dns, dbname=dbname, user=user_email,
                        password=cred.token, sslmode='require')
with conn.cursor() as cur:
    cur.execute(f'CREATE SCHEMA IF NOT EXISTS "{schema}"')
    cur.execute(f"""
    CREATE TABLE IF NOT EXISTS "{schema}"."cat_reviews" (
        review_id TEXT, order_id TEXT, source TEXT, reviewer TEXT,
        review_date DATE, original_level_1 TEXT, original_level_2 TEXT,
        original_level_3 TEXT, reviewed_level_1 TEXT, reviewed_level_2 TEXT,
        reviewed_level_3 TEXT, review_status TEXT, comments TEXT,
        created_at TIMESTAMPTZ
    )
    """)
    conn.commit()
conn.close()
print(f'Native cat_reviews ready in {instance_name}/{dbname}/{schema}')